# Intel Image Classification — Final Project

This notebook classifies natural scene images from the **Intel Image Classification** Kaggle dataset into 6 categories:
> `buildings`, `forest`, `glacier`, `mountain`, `sea`, `street`

It follows the standard ML workflow:
1. Download & explore the dataset
2. Build an input pipeline
3. Build a baseline CNN model
4. Train & visualize results
5. Address overfitting (data augmentation + dropout)
6. Retrain & evaluate
7. Predict on new images
8. (Optional) Convert to TensorFlow Lite

**Reference:** Adapted from the [TensorFlow Image Classification tutorial](https://www.tensorflow.org/tutorials/images/classification).

## 1. Setup



In [ ]:
!pip install kagglehub -q

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pathlib
import PIL
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

print("TensorFlow version:", tf.__version__)

## 2. Download the Dataset

use `kagglehub` to download the Intel Image Classification dataset.
The dataset contains aroundv 25,000 images (150×150 px) across 6 scene categories.



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("puneet6060/intel-image-classification")
print("Path to dataset files:", path)

In [ ]:
import os

data_root = pathlib.Path(path)

# The dataset ships with seg_train and seg_test directories
train_dir = data_root / "seg_train" / "seg_train"
test_dir  = data_root / "seg_test"  / "seg_test"

# Verify directories exist
print("Train dir:", train_dir)
print("Test dir: ", test_dir)

# Count images
train_count = len(list(train_dir.glob('*/*.jpg')))
test_count  = len(list(test_dir.glob('*/*.jpg')))
print(f"\nTraining images : {train_count}")
print(f"Test images     : {test_count}")

## 3. Explore the Dataset


In [ ]:
# Class names from subdirectory names
class_names = sorted([item.name for item in train_dir.iterdir() if item.is_dir()])
print("Classes:", class_names)
print("Number of classes:", len(class_names))

In [ ]:
# Display one sample image per class
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for i, cls in enumerate(class_names):
    sample_images = list((train_dir / cls).glob('*.jpg'))
    img = PIL.Image.open(str(sample_images[0]))
    axes[i].imshow(img)
    axes[i].set_title(cls, fontsize=14, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Intel Image Classification — One Sample per Class', fontsize=16)
plt.tight_layout()
plt.show()

## 4. Build the Input Pipeline

We use `tf.keras.utils.image_dataset_from_directory` to load images efficiently.

- Images are resized to **150×150** (matching the original resolution)
- Batch size: **32**
- The training set is split 80/20 into train/validation

In [ ]:
IMG_HEIGHT = 150
IMG_WIDTH  = 150
BATCH_SIZE = 32

# Training dataset (80% of seg_train)
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)

# Validation dataset (20% of seg_train)
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)

# Held-out test dataset (seg_test)
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    seed=42,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE
)

# Confirm class names match
print("Class names:", train_ds.class_names)

In [ ]:
# Visualize the first 9 training images
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i]], fontsize=12)
        plt.axis('off')
plt.suptitle('Sample Training Images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Configure datasets for performance
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 5. Baseline CNN Model

We build a simple Sequential CNN with:
- **3 Conv2D + MaxPooling2D** blocks (increasing filters: 16 → 32 → 64)
- **Flatten** → **Dense(128, relu)** → **Dense(6)** (one per class)
- Pixel normalization via `Rescaling(1./255)` as the first layer

This mirrors the baseline architecture from the TensorFlow tutorial.

In [ ]:
num_classes = len(class_names)  # 6

baseline_model = Sequential([
    # Normalize pixel values from [0, 255] → [0, 1]
    layers.Rescaling(1./255, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),

    # layer 1
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),

    # layer 2
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),

    # layer 3
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),

    # Classifier head
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation = 'softmax')
], name='baseline_cnn')

baseline_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

baseline_model.summary()

In [ ]:
EPOCHS_BASELINE = 10

history_baseline = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_BASELINE
)

In [ ]:
def plot_history(history, title_suffix=''):
    """Plot training & validation accuracy and loss."""
    acc     = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss     = history.history['loss']
    val_loss = history.history['val_loss']
    epochs_range = range(len(acc))

    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, acc,     label='Training Accuracy')
    plt.plot(epochs_range, val_acc, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.title(f'Accuracy {title_suffix}')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss,     label='Training Loss')
    plt.plot(epochs_range, val_loss, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.title(f'Loss {title_suffix}')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')

    plt.tight_layout()
    plt.show()

plot_history(history_baseline, title_suffix='— Baseline Model')

## 6. Overfitting Analysis
We address this with two techniqu: 
1. **Data Augmentation** — randomly transform images during training to expose the model to more variation
2. **Dropout** — randomly zero out neurons during training to prevent co-adaptation

## 7. Data Augmentation

We apply random horizontal flips, rotations, and zooms. These layers run on GPU and are only active during training.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal", input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name='data_augmentation')

# Visualize augmented examples
plt.figure(figsize=(10, 10))
for images, _ in train_ds.take(1):
    for i in range(9):
        augmented = data_augmentation(images)
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(augmented[0].numpy().astype('uint8'))
        plt.axis('off')
plt.suptitle('Augmented Training Images (same source image, different transforms)', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Improved Model (Augmentation + Dropout)

We prepend the augmentation pipeline and add a **Dropout(0.2)** layer before the dense head.
We also increase training to **15 epochs** to allow the regularized model to learn more slowly.

In [ ]:
improved_model = Sequential([
    # Data augmentation (active only during training)
    data_augmentation,

    # Normalize pixel values
    layers.Rescaling(1./255),

    # layer 1
    layers.Conv2D(16, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),

    # layer 2
    layers.Conv2D(32, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),

    # layer 3
    layers.Conv2D(64, 3, padding='same', activation='relu'),
    layers.MaxPooling2D(),

    # Regularization: randomly drop 20% of neurons
    layers.Dropout(0.2),

    # Classifier head
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes,activation = 'softmax',name='outputs')
], name='improved_cnn')

improved_model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

improved_model.summary()

In [ ]:
EPOCHS_IMPROVED = 10

history_improved = improved_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_IMPROVED
)

In [ ]:
plot_history(history_improved, title_suffix='— Improved Model (Augmentation + Dropout)')

## 9. Evaluate on the Test Set

We now evaluate the improved model on the **held-out test set** (`seg_test`), which was never seen during training.

In [ ]:
test_loss, test_accuracy = improved_model.evaluate(test_ds)
print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy * 100:.2f}%")

## 10. Confusion Matrix

A confusion matrix shows how well the model distinguishes between each class.

In [ ]:
import sklearn.metrics as skm

# Collect all true labels and predictions from test set
y_true, y_pred = [], []

for images, labels in test_ds:
    predictions = improved_model(images, training=False)
    predicted_classes = tf.argmax(predictions, axis=1).numpy()
    y_true.extend(labels.numpy())
    y_pred.extend(predicted_classes)

# Confusion matrix
cm = skm.confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)

tick_marks = np.arange(num_classes)
ax.set_xticks(tick_marks)
ax.set_yticks(tick_marks)
ax.set_xticklabels(class_names, rotation=45, ha='right')
ax.set_yticklabels(class_names)

# Annotate cells
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha='center', va='center',
                color='white' if cm[i, j] > thresh else 'black')

ax.set_ylabel('True Label', fontsize=13)
ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_title('Confusion Matrix — Test Set', fontsize=14)
plt.tight_layout()
plt.show()

# Classification report
print("\nClassification Report:")
print(skm.classification_report(y_true, y_pred, target_names=class_names))

## 11. Predict on New Images

Use the improved model to classify a single image from the test set.

In [ ]:
# Pick a sample image from the test directory
sample_class = class_names[0]  # e.g., 'buildings'
sample_images = list((test_dir / sample_class).glob('*.jpg'))
sample_path = str(sample_images[5])

# Load and preprocess
img = tf.keras.utils.load_img(sample_path, target_size=(IMG_HEIGHT, IMG_WIDTH))
img_array = tf.keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0)

# Predict
predictions = improved_model.predict(img_array)
score = tf.nn.softmax(predictions[0])

predicted_class = class_names[np.argmax(score)]
confidence = 100 * np.max(score)

# Display result
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.title(f"Predicted: {predicted_class} ({confidence:.1f}%)\nActual: {sample_class}",
          fontsize=13,
          color='green' if predicted_class == sample_class else 'red')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Predicted class : {predicted_class} ({confidence:.2f}% confidence)")
print(f"Actual class    : {sample_class}")

In [ ]:
# Save model as H5
improved_model.save('intel_classifier.h5')
print(f"Model saved. Size: {os.path.getsize('intel_classifier.h5') / 1024:.1f} KB")

## Summary

| Step | Details |
|------|---------|
| **Dataset** | Intel Image Classification (Kaggle) — 6 classes, ~25,000 images |
| **Image size** | 150×150 px |
| **Baseline model** | 3× Conv2D + MaxPool, Dense(128), Dense(6) — 10 epochs |
| **Problem identified** | Overfitting: large gap between training and validation accuracy |
| **Solution** | Data augmentation (flip, rotate, zoom) + Dropout(0.2) |
| **Improved model** | Same architecture + augmentation + dropout — 15 epochs |
| **Evaluation** | Confusion matrix + classification report on held-out `seg_test` |
| **Deployment** | TensorFlow Lite conversion for mobile/embedded inference |
